# Verify one ZM channel — start with FIR only

Run one cell at a time and stop at **Stop here: review FIR**. No AcqKnowledge comparison has been run yet.

For this first test, save a **separate copy** of one recording with its unchanged **raw ZM** and the **FIR result**. Use FIR 28–500 Hz, Blackman −61 dB, optimized coefficient count, and the entire waveform. Do not crop, resample or change units. Comb and RMS channels are not needed yet.

We compare the original raw samples with the raw channel retained in that copy, then compare Python FIR against AcqKnowledge FIR, sample for sample. Sample zero must represent the same instant. Errors use the original channel's units; sample indices are zero-based.

The function below is a standalone snapshot from `notebook.ipynb`; its TODOs remain unresolved. No acceptance threshold is assumed and no files are written. Future comb/RMS cells remain below the FIR stopping point.

In [17]:
from pathlib import Path
import bioread
import numpy as np
import pandas as pd
from scipy import signal
import matplotlib.pyplot as plt
import mne

## 1. Read the original and AcqKnowledge reference
Replace the two paths, then run the loading cell.

In [2]:
reference_file = Path("TEST_fEMG_div_4SlowRate_raw.acq")

In [3]:
reference_recording = bioread.read(reference_file)

In [4]:
pd.DataFrame([
    {"file": label, "index": i, "name": ch.name, "units": ch.units,
    "rate_hz": ch.samples_per_second, "samples": len(ch.data)}
    for label, recording in [("reference", reference_recording)]
    for i, ch in enumerate(recording.channels)
])

,file,index,name,units,rate_hz,samples
0,reference,0,ZM - FIR,mV,2000.0,805412
1,reference,1,ZM - Comb,mV,2000.0,805412
2,reference,2,ZM - EMG100C,mV,2000.0,805412
3,reference,3,CS - EMG100C,mV,2000.0,805412
4,reference,4,ZM - Integrate,mV,2000.0,805412


Choose the **original raw ZM index**, then the **raw ZM and FIR indices in the reference file**. Use the Python `index` column above. Only these three choices are needed for FIR.

In [5]:
raw_index = 2
reference_indices_fir = 0

In [6]:
raw_channel = reference_recording.channels[raw_index]
reference_channel_fir = reference_recording.channels[reference_indices_fir]


In [7]:
raw_channel

Channel ZM - EMG100C: 805412 samples, 2000.0 samples/sec, loaded: True

In [8]:
reference_channel_fir

Channel ZM - FIR: 805412 samples, 2000.0 samples/sec, loaded: True

In [9]:
def compare(actual, expected, sampling_rate):
    actual = np.asarray(actual, dtype=float)
    expected = np.asarray(expected, dtype=float)

    if (
        actual.ndim != 1
        or expected.ndim != 1
        or actual.shape != expected.shape
        or actual.size == 0
    ):
        raise ValueError("Signals must be 1D with the same nonzero length.")

    if not (np.isfinite(actual).all() and np.isfinite(expected).all()):
        raise ValueError("Signals contain missing or infinite samples.")

    if not np.isfinite(sampling_rate) or sampling_rate <= 0:
        raise ValueError("sampling_rate must be positive and finite.")

    difference = actual - expected
    worst = int(np.argmax(np.abs(difference)))
    rmse = np.sqrt(np.mean(difference**2))
    reference_rms = np.sqrt(np.mean(expected**2))

    return pd.Series({
        "max_absolute_error": np.max(np.abs(difference)),
        "rmse": rmse,
        "relative_rmse_percent": (
            100 * rmse / reference_rms if reference_rms > 0 else np.nan
        ),
        "mean_error": np.mean(difference),
        "worst_sample": worst,
        "worst_time_seconds": worst / sampling_rate,
    })

## 2. FIR

In [10]:
raw = np.asarray(raw_channel.data, dtype=float)
reference = np.asarray(reference_channel_fir.data, dtype=float)

### Parameters

In [12]:
sampling_rate = raw_channel.samples_per_second

low_hz = 28.0
high_hz = 500.0
num_taps = 285

fir_scale = False
fir_window = [0.44959, 0.49364, 0.05677, 0.0]
fir_padding = "edge"

In [13]:
assert raw.ndim == reference.ndim == 1, "Use one-dimensional channel arrays."
assert raw.size > 0 and reference.size > 0, "An array is empty."
assert np.isfinite(raw).all(), "Raw signal contains NaN or infinity."
assert np.isfinite(reference).all(), "Reference contains NaN or infinity."
assert np.isfinite(sampling_rate) and sampling_rate > 0, "Check the sampling rate."

print(f"Sampling rate:       {sampling_rate} Hz")
print(f"Raw samples:         {raw.size}")
print(f"Reference samples:   {reference.size}")
print(f"Raw duration:        {raw.size / sampling_rate:.6f} seconds")
print(f"Same length:         {raw.size == reference.size}")
print(f"Nyquist frequency:   {sampling_rate / 2} Hz")

Sampling rate:       2000.0 Hz
Raw samples:         805412
Reference samples:   805412
Raw duration:        402.706000 seconds
Same length:         True
Nyquist frequency:   1000.0 Hz


In [14]:
# Reproduce the calculation in your original function.
calculated_count = 4 * sampling_rate / low_hz
num_taps = int(calculated_count)

# Reproduce its adjustment to an odd number.
if num_taps % 2 == 0:
    num_taps += 1

half_length = (num_taps - 1) // 2

print("Count before rounding:", calculated_count)
print("Count used by your Python function:", num_taps)
print(f"Half-length:            {half_length} samples")
print(f"Half-length in time:    {1000 * half_length / sampling_rate:.2f} ms")

if high_hz >= sampling_rate / 2:
    raise ValueError(
        "This implementation requires a sampling rate greater than "
        "1000 Hz for a 500 Hz upper cutoff."
    )

Count before rounding: 285.7142857142857
Count used by your Python function: 285
Half-length:            142 samples
Half-length in time:    71.00 ms


In [ ]:
# Reproduce the initial filter-design step from your function.
base_coefficients = signal.firwin(
    numtaps=num_taps,
    cutoff=[low_hz, high_hz],
    fs=sampling_rate,
    pass_zero=False,
    window="boxcar",
    scale=False,
)

print("Number of coefficients:", base_coefficients.size)
print("All finite:", np.isfinite(base_coefficients).all())
print(
    "Symmetric:",
    np.allclose(base_coefficients, base_coefficients[::-1]),
)
print("Center coefficient:", base_coefficients[half_length])

offset_ms = (np.arange(num_taps) - half_length) / sampling_rate * 1000

plt.figure(figsize=(10, 3))
plt.plot(offset_ms, base_coefficients, linewidth=0.8)
plt.xlabel("Offset from filter center (ms)")
plt.ylabel("Coefficient")
plt.title("FIR coefficients before applying the Blackman window")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
window = signal.windows.general_cosine(
    num_taps,
    fir_window,
    sym=True,
)

# Keep the unwindowed coefficients for comparison.
windowed_coefficients = base_coefficients * window

print("Window center:", window[half_length])
print("Window endpoints:", window[0], window[-1])
print("All finite:", np.isfinite(windowed_coefficients).all())
print(
    "Symmetric:",
    np.allclose(windowed_coefficients, windowed_coefficients[::-1]),
)
print("Center coefficient:", windowed_coefficients[half_length])

plt.figure(figsize=(10, 3))
plt.plot(offset_ms, window)
plt.xlabel("Offset from filter center (ms)")
plt.ylabel("Window weight")
plt.title("Window applied to the FIR coefficients")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
center_freq = (low_hz + high_hz) / 2

_, response_before = signal.freqz(
    windowed_coefficients,
    worN=[center_freq],
    fs=sampling_rate,
)

gain_before = abs(response_before[0])

if not np.isfinite(gain_before) or gain_before == 0:
    raise ValueError("Cannot normalize: gain is zero or nonfinite.")

# Preserve the previous coefficients.
coefficients = windowed_coefficients / gain_before

_, response_after = signal.freqz(
    coefficients,
    worN=[center_freq],
    fs=sampling_rate,
)

print("Normalization frequency:", center_freq, "Hz")
print("Gain before normalization:", gain_before)
print("Gain after normalization:", abs(response_after[0]))
print("Largest coefficient change:",
      np.max(np.abs(coefficients - windowed_coefficients)))

In [ ]:
# Reproduce your original function's filtering step.
padded = np.pad(raw, (half_length, half_length), mode="edge")
python_filtered = signal.convolve(padded, coefficients, mode="valid")

assert python_filtered.shape == reference.shape

# Compare both the entire recording and the interior.
# The interior excludes samples that depend on our reflected padding.
interior = slice(half_length, -half_length)

for label, selection in [
    ("Full recording", slice(None)),
    ("Interior", interior),
]:
    expected = reference[selection]
    actual = python_filtered[selection]
    difference = actual - expected

    rmse = np.sqrt(np.mean(difference**2))
    reference_rms = np.sqrt(np.mean(expected**2))

    print(f"\n{label}")
    print(f"RMSE:               {rmse:.10g}")
    print(f"Maximum abs error:  {np.max(np.abs(difference)):.10g}")

    if reference_rms > 0:
        print(f"RMSE / reference RMS: {100 * rmse / reference_rms:.4f}%")

In [ ]:
edge_mask = np.zeros(raw.size, dtype=bool)
edge_mask[:half_length] = True
edge_mask[-half_length:] = True

for mode in ["reflect", "symmetric", "edge", "constant"]:
    # "constant" uses zero padding by default.
    padded_test = np.pad(
        raw,
        (half_length, half_length),
        mode=mode,
    )

    candidate = signal.convolve(
        padded_test,
        coefficients,
        mode="valid",
    )

    difference = candidate - reference

    full_rmse = np.sqrt(np.mean(difference**2))
    edge_rmse = np.sqrt(np.mean(difference[edge_mask]**2))
    max_error = np.max(np.abs(difference))

    print(
        f"{mode:10s} | Full RMSE: {full_rmse:.10g} "
        f"| Edge RMSE: {edge_rmse:.10g} "
        f"| Max error: {max_error:.10g}"
    )

In [ ]:
# Change these to inspect another part of the recording.
start_seconds = 10
duration_seconds = 2

start = int(start_seconds * sampling_rate)
stop = min(
    start + int(duration_seconds * sampling_rate),
    len(raw),
)
time = np.arange(start, stop) / sampling_rate

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Overlay in original units: mV.
axes[0].plot(
    time, reference[start:stop],
    label="AcqKnowledge FIR", linewidth=2, alpha=0.7,
)
axes[0].plot(
    time, python_filtered[start:stop],
    label="Python FIR", linestyle="--", linewidth=1.2,
)
axes[0].set_ylabel("Amplitude (mV)")
axes[0].set_title("AcqKnowledge vs Python — FIR 28–500 Hz")
axes[0].legend()

# Convert the difference from mV to µV for easier reading.
difference_uv = (
    python_filtered[start:stop] - reference[start:stop]
) * 1000

axes[1].plot(time, difference_uv, color="tab:red", linewidth=1)
axes[1].axhline(0, color="black", linewidth=0.7)
axes[1].set_ylabel("Difference (µV)")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_title("Python − AcqKnowledge")

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
padded = np.pad(raw, (half_length, half_length), mode="edge")

for name, taps in [
    ("Normalized", coefficients),
    ("Without normalization", windowed_coefficients),
]:
    candidate = signal.convolve(padded, taps, mode="valid")
    difference = candidate - reference

    full_rmse = np.sqrt(np.mean(difference**2))
    interior_rmse = np.sqrt(np.mean(difference[interior]**2))

    print(f"\n{name}")
    print(f"Full RMSE:     {full_rmse:.10g}")
    print(f"Interior RMSE: {interior_rmse:.10g}")
    print(f"Maximum error: {np.max(np.abs(difference)):.10g}")

In [ ]:
coefficients = windowed_coefficients.copy()

padded = np.pad(raw, (half_length, half_length), mode="edge")
python_filtered = signal.convolve(padded, coefficients, mode="valid")

In [ ]:
normalized_coefficients = windowed_coefficients / gain_before

for name, taps in [
    ("Normalized", normalized_coefficients),
    ("Without normalization", windowed_coefficients),
]:
    candidate = signal.convolve(padded, taps, mode="valid")
    difference = candidate - reference

    print(f"\n{name}")
    print(f"Full RMSE:     {np.sqrt(np.mean(difference**2)):.10g}")
    print(
        f"Interior RMSE: "
        f"{np.sqrt(np.mean(difference[interior]**2)):.10g}"
    )
    print(f"Maximum error: {np.max(np.abs(difference)):.10g}")

## Functions

In [15]:
def biopac_fir_bandpass(
    data,
    sampling_rate,
    low_hz=28.0,
    high_hz=500.0,
    num_taps=285,
):
    """
    Approximate the AcqKnowledge FIR band-pass tested in this project.

    Uses a symmetric custom Blackman window, no gain normalization,
    edge padding, and centered convolution.

    The default 285 coefficients were confirmed in AcqKnowledge for
    the tested 2000 Hz recording. For other settings, supply the
    coefficient count shown in AcqKnowledge.

    Returns a new 1D array with the same length and units as data.
    """
    x = np.asarray(data, dtype=float)

    if x.ndim != 1 or x.size == 0:
        raise ValueError("data must be a nonempty, one-dimensional array.")
    if not np.isfinite(x).all():
        raise ValueError("data contains NaN or infinite values.")
    if not np.isfinite(sampling_rate) or sampling_rate <= 0:
        raise ValueError("sampling_rate must be positive and finite.")
    if not 0 < low_hz < high_hz < sampling_rate / 2:
        raise ValueError(
            "Cutoffs must satisfy: 0 < low_hz < high_hz < sampling_rate / 2."
        )
    if (
        isinstance(num_taps, (bool, np.bool_))
        or not isinstance(num_taps, (int, np.integer))
        or num_taps < 3
        or num_taps % 2 == 0
    ):
        raise ValueError("num_taps must be an odd integer of at least 3.")

    # Design the base FIR without gain normalization.
    coefficients = signal.firwin(
        numtaps=num_taps,
        cutoff=[low_hz, high_hz],
        fs=sampling_rate,
        pass_zero=False,
        window="boxcar",
        scale=False,
    )

    # Apply the custom window used in our comparison.
    window = signal.windows.general_cosine(
        num_taps,
        [0.44959, 0.49364, 0.05677, 0.0],
        sym=True,
    )
    coefficients *= window

    # Repeat endpoint values and center the filtered output.
    half_length = (num_taps - 1) // 2
    padded = np.pad(x, (half_length, half_length), mode="edge")

    return signal.convolve(padded, coefficients, mode="valid")

In [18]:
def neurokit_mne_fir_bandpass(
    data,
    sampling_rate,
    low_hz=28.0,
    high_hz=500.0,
    filter_length="auto",
    fir_window="hamming",
    phase="zero-double",
    padding="reflect_limited",
):
    """FIR band-pass using NeuroKit2's MNE configuration."""
    x = np.asarray(data, dtype=float)

    if x.ndim != 1 or x.size == 0:
        raise ValueError("data must be a nonempty 1D array.")
    if not np.isfinite(x).all():
        raise ValueError("data contains NaN or infinite values.")
    if not np.isfinite(sampling_rate) or sampling_rate <= 0:
        raise ValueError("sampling_rate must be positive and finite.")
    if not 0 < low_hz < high_hz < sampling_rate / 2:
        raise ValueError(
            "Require 0 < low_hz < high_hz < sampling_rate / 2."
        )

    return mne.filter.filter_data(
        data=x,
        sfreq=sampling_rate,
        l_freq=low_hz,
        h_freq=high_hz,
        method="fir",
        filter_length=filter_length,
        l_trans_bandwidth="auto",
        h_trans_bandwidth="auto",
        fir_window=fir_window,
        fir_design="firwin",
        phase=phase,
        pad=padding,
        copy=True,
        verbose=True,
    )

In [16]:
python_filtered = biopac_fir_bandpass(
    raw,
    sampling_rate=2000.0,
    num_taps=285,
)

In [24]:
fir_mne_blackman = neurokit_mne_fir_bandpass(
    raw,
    sampling_rate=sampling_rate,
    low_hz=28.0,
    high_hz=500.0,
    fir_window="blackman",  # MNE's standard Blackman, not our custom weights
    phase="zero",
    padding="edge",
    filter_length="auto",
)

Setting up band-pass filter from 28 - 5e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Blackman window with 0.0017 passband ripple and 74 dB stopband attenuation
- Lower passband edge: 28.00
- Lower transition bandwidth: 7.00 Hz (-6 dB cutoff frequency: 24.50 Hz)
- Upper passband edge: 500.00 Hz
- Upper transition bandwidth: 125.00 Hz (-6 dB cutoff frequency: 562.50 Hz)
- Filter length: 1429 samples (0.715 s)



In [19]:
fir_mne = neurokit_mne_fir_bandpass(
    raw,
    sampling_rate=sampling_rate,
)

Setting up band-pass filter from 28 - 5e+02 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 28.00
- Lower transition bandwidth: 7.00 Hz (-12 dB cutoff frequency: 24.50 Hz)
- Upper passband edge: 500.00 Hz
- Upper transition bandwidth: 125.00 Hz (-12 dB cutoff frequency: 562.50 Hz)
- Filter length: 943 samples (0.471 s)



In [22]:
print("Our FIR")
compare(python_filtered, reference, sampling_rate)

Our FIR


max_absolute_error       1.730032e-04
rmse                     1.220032e-05
relative_rmse_percent    5.785320e-03
mean_error               4.826925e-10
worst_sample             3.877020e+05
worst_time_seconds       1.938510e+02
dtype: float64

In [23]:
print("\nNeuroKit2 / MNE FIR")
compare(fir_mne, reference, sampling_rate)


NeuroKit2 / MNE FIR


max_absolute_error       0.087965
rmse                     0.002817
relative_rmse_percent    1.335790
mean_error               0.000002
worst_sample             1.000000
worst_time_seconds       0.000500
dtype: float64

In [25]:
print("\nNeuroKit2 / MNE FIR")
compare(fir_mne_blackman, reference, sampling_rate)


NeuroKit2 / MNE FIR


max_absolute_error       1.402149e-02
rmse                     3.685517e-03
relative_rmse_percent    1.747650e+00
mean_error              -4.401864e-08
worst_sample             3.500000e+01
worst_time_seconds       1.750000e-02
dtype: float64

In [ ]:
def biopac_comb_band_stop(data, sampling_rate, base_hz=60.0, q=5.0):
    """
    Approximates the specified AcqKnowledge Comb Band Stop filter settings.
    Equivalence requires sample-by-sample numerical comparison with AcqKnowledge.

    Apply to the band-pass filtered signal. Cascades IIR notch filters at
    base_hz and its integer multiples strictly below Nyquist.

    Each notch uses the same Q, with bandwidth = center_frequency / Q.
    Returns a new array with the same length as the input.

    Reference:
        AcqKnowledge 5 Software Guide, pp. 337–338: Comb Band Stop.
    """
    nyquist = sampling_rate / 2.0

    if base_hz <= 0 or base_hz >= nyquist:
        raise ValueError(
            f"base_hz ({base_hz} Hz) must be positive and strictly less than "
            f"the Nyquist frequency ({nyquist} Hz)."
        )
    if q <= 0:
        raise ValueError(f"q ({q}) must be positive.")

    # Determine the highest harmonic below Nyquist.
    k_max = int(np.floor(nyquist / base_hz))

    # Exclude a harmonic that falls at or above Nyquist.
    if k_max * base_hz >= nyquist:
        k_max -= 1

    filtered = np.array(data, dtype=float, copy=True)

    # Apply notch filters in ascending frequency order.
    # TODO: verify numerical agreement with AcqKnowledge.
    for k in range(1, k_max + 1):
        frequency = k * base_hz

        # BIOPAC documents a cascade of IIR band-stop filters.
        # SciPy uses second-order notches with 3 numerator and 3 denominator coefficients.
        # TODO: verify coefficients against AcqKnowledge using an impulse response.
        b, a = signal.iirnotch(frequency, q, fs=sampling_rate)

        # Apply one forward pass with zero initial state.
        # TODO: verify AcqKnowledge's processing direction and initial conditions.
        # Forward-backward filtering would also change the magnitude response.
        filtered = signal.lfilter(b, a, filtered)

    return filtered

In [ ]:
# Channel 0: AcqKnowledge FIR output.
comb_input = np.asarray(
    reference_recording.channels[0].data,
    dtype=float,
)

# Channel 1: AcqKnowledge FIR → comb output.
comb_reference = np.asarray(
    reference_recording.channels[1].data,
    dtype=float,
)

python_comb = biopac_comb_band_stop(
    comb_input,
    sampling_rate=sampling_rate,
    base_hz=60.0,
    q=5.0,
)

compare(python_comb, comb_reference, sampling_rate)

In [ ]:
for skip_seconds in [0, 0.1, 0.5, 1, 2, 5]:
    start = int(skip_seconds * sampling_rate)

    result = compare(
        python_comb[start:],
        comb_reference[start:],
        sampling_rate,
    )

    print(
        f"Skip first {skip_seconds:4.1f} s | "
        f"RMSE: {result['rmse']:.8g} | "
        f"Relative RMSE: {result['relative_rmse_percent']:.4f}% | "
        f"Max error: {result['max_absolute_error']:.8g}"
    )

In [ ]:
# Compare the interior, excluding 1 second at each end.
trim = int(sampling_rate)

result = compare(
    python_comb[trim:-trim],
    comb_reference[trim:-trim],
    sampling_rate,
)

print(result)

# Locate the largest remaining difference in original recording time.
worst_original_sample = trim + int(result["worst_sample"])

print(
    "\nWorst interior error at:",
    worst_original_sample / sampling_rate,
    "seconds",
)

In [ ]:
trim = int(sampling_rate)
max_lag = 10

start = trim + max_lag
stop = len(python_comb) - trim - max_lag

actual = python_comb[start:stop]

for lag in range(-max_lag, max_lag + 1):
    expected = comb_reference[start + lag : stop + lag]
    difference = actual - expected

    rmse = np.sqrt(np.mean(difference**2))
    reference_rms = np.sqrt(np.mean(expected**2))

    print(
        f"Lag {lag:+3d} samples "
        f"({1000 * lag / sampling_rate:+5.1f} ms) | "
        f"Relative RMSE: {100 * rmse / reference_rms:.4f}%"
    )

In [ ]:
# Diagnostic alignment only; preserve python_comb unchanged.
trim = int(sampling_rate)

actual = python_comb[trim + 1 : -trim]
expected = comb_reference[trim : -trim - 1]

# Best-fitting scalar: expected ≈ gain * actual.
gain = np.dot(actual, expected) / np.dot(actual, actual)

reference_rms = np.sqrt(np.mean(expected**2))
error_before = actual - expected
error_after = gain * actual - expected

print("Best-fitting gain:", gain)
print(
    "Aligned relative RMSE:",
    100 * np.sqrt(np.mean(error_before**2)) / reference_rms,
    "%",
)
print(
    "Aligned + scaled relative RMSE:",
    100 * np.sqrt(np.mean(error_after**2)) / reference_rms,
    "%",
)

In [ ]:
for start_seconds in [10, 100, 190, 300]:
    start = int(start_seconds * sampling_rate)
    stop = start + int(10 * sampling_rate)

    actual = python_comb[start:stop]
    results = []

    for lag in [-2, -1, 0]:
        expected = comb_reference[start + lag : stop + lag]

        relative_rmse = (
            100
            * np.sqrt(np.mean((actual - expected)**2))
            / np.sqrt(np.mean(expected**2))
        )
        results.append(f"lag {lag:+d}: {relative_rmse:.3f}%")

    print(f"{start_seconds}–{start_seconds + 10} seconds | "
          + " | ".join(results))

In [ ]:
# Inspect one second beginning at 10 seconds.
start = int(10 * sampling_rate)
stop = start + int(sampling_rate)
time = np.arange(start, stop) / sampling_rate

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(time, reference[start:stop],
             label="AcqKnowledge", linewidth=1)
axes[0].plot(time, python_filtered[start:stop],
             label="Python", linewidth=0.8, alpha=0.75)
axes[0].set_ylabel("Amplitude")
axes[0].legend()

axes[1].plot(
    time,
    python_filtered[start:stop] - reference[start:stop],
    linewidth=0.8,
)
axes[1].set_ylabel("Python − reference")
axes[1].set_xlabel("Time (seconds)")

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Inspect the same interior region in all three signals.
selection = slice(half_length, -half_length)

for name, values in [
    ("Raw", raw[selection]),
    ("Python FIR", python_filtered[selection]),
    ("AcqKnowledge FIR", reference[selection]),
]:
    print(f"\n{name}")
    print(f"Mean:               {np.mean(values):.10g}")
    print(f"Standard deviation: {np.std(values):.10g}")
    print(f"RMS:                {np.sqrt(np.mean(values**2)):.10g}")
    print(f"Minimum:            {np.min(values):.10g}")
    print(f"Maximum:            {np.max(values):.10g}")

print("\nSum of FIR coefficients:", np.sum(coefficients))

In [ ]:
p = python_filtered[interior]
r = reference[interior]

# Remove means for this diagnostic.
p_centered = p - np.mean(p)
r_centered = r - np.mean(r)

# Best-fitting multiplier: reference ≈ multiplier * Python.
multiplier = np.dot(p_centered, r_centered) / np.dot(
    p_centered, p_centered
)

correlation = np.corrcoef(p_centered, r_centered)[0, 1]
residual = r_centered - multiplier * p_centered

relative_error = (
    np.sqrt(np.mean(residual**2))
    / np.sqrt(np.mean(r_centered**2))
)

print(f"Correlation at zero lag:       {correlation:.6f}")
print(f"Best-fitting multiplier:       {multiplier:.10g}")
print(f"Residual / reference RMS:      {100 * relative_error:.4f}%")

In [ ]:
# Use an interior segment, away from recording boundaries.
start = int(10 * sampling_rate)
stop = int(30 * sampling_rate)
max_lag = int(0.5 * sampling_rate)

p = python_filtered[start:stop]
r = reference[start - max_lag : stop + max_lag]

p = p - p.mean()

# Compare p with every equally sized window of r.
numerator = signal.correlate(r, p, mode="valid", method="fft")

# Mean-centered energy for each reference window.
n = p.size
sum_r = np.concatenate(([0.0], np.cumsum(r)))
sum_r2 = np.concatenate(([0.0], np.cumsum(r**2)))

window_sum = sum_r[n:] - sum_r[:-n]
window_sum2 = sum_r2[n:] - sum_r2[:-n]
reference_energy = np.maximum(
    window_sum2 - window_sum**2 / n, 0
)

denominator = np.sqrt(np.dot(p, p) * reference_energy)

correlations = np.divide(
    numerator,
    denominator,
    out=np.full_like(numerator, np.nan),
    where=denominator > 0,
)

lags = np.arange(-max_lag, max_lag + 1)
best = np.nanargmax(np.abs(correlations))

print("Best lag:", lags[best], "samples")
print("Best lag:", 1000 * lags[best] / sampling_rate, "ms")
print("Correlation at best lag:", correlations[best])
print("Correlation at zero lag:", correlations[max_lag])

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

for name, values in [
    ("Raw", raw),
    ("Python FIR", python_filtered),
    ("AcqKnowledge FIR", reference),
]:
    frequencies, power = signal.welch(
        values[interior],
        fs=sampling_rate,
        nperseg=8192,
        detrend="constant",
        scaling="density",
    )

    ax.semilogy(
        frequencies,
        np.maximum(power, np.finfo(float).tiny),
        label=name,
        alpha=0.8,
    )

    # Strongest non-DC frequency.
    peak = 1 + np.argmax(power[1:])
    print(f"{name}: strongest frequency = {frequencies[peak]:.2f} Hz")

ax.axvline(low_hz, color="gray", linestyle="--", label="28 / 500 Hz cutoffs")
ax.axvline(high_hz, color="gray", linestyle="--")
ax.set_xlim(0, sampling_rate / 2)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power spectral density (input units²/Hz)")
ax.set_title("Frequency content before and after filtering")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
for name, values in [
    ("Raw", raw),
    ("Python FIR", python_filtered),
    ("AcqKnowledge reference", reference),
]:
    frequencies, power = signal.welch(
        values[interior],
        fs=sampling_rate,
        nperseg=8192,
        detrend="constant",
        scaling="density",
    )

    df = frequencies[1] - frequencies[0]

    near_60 = (frequencies >= 58) & (frequencies <= 62)
    passband = (frequencies >= 28) & (frequencies <= 500)
    other_passband = passband & ~near_60

    rms_60 = np.sqrt(np.sum(power[near_60]) * df)
    rms_other = np.sqrt(np.sum(power[other_passband]) * df)

    print(f"\n{name}")
    print(f"58–62 Hz RMS estimate:           {rms_60:.8g}")
    print(f"Remaining passband RMS estimate: {rms_other:.8g}")

In [ ]:

# Create the rectangular-window base; apply Blackman separately below.
coefficients = signal.firwin(num_taps, [low_hz, high_hz], fs=sampling_rate, pass_zero=False, window="boxcar", scale=False,)

coefficients *= signal.windows.general_cosine(num_taps, fir_window)

In [ ]:
# Set filter gain to 1.0 at the passband center.
# TODO: verify normalization using the interior amplitude of a filtered sine wave.
center_freq = (low_hz + high_hz) / 2.0
_, gain = signal.freqz(coefficients, worN=[center_freq], fs=sampling_rate)
coefficients /= abs(gain[0])

# Padding preserves output length. Centered convolution compensates
# the odd-length filter's half-window delay.
left = (num_taps - 1) // 2
right = num_taps // 2

# TODO: verify reflected boundaries by comparing filtered ramps in both programs.
padded = np.pad(data, (left, right), mode=fir_padding)

# TODO: compare filtered impulses sample by sample to verify timing.
signal.convolve(padded, coefficients, mode="valid")

In [ ]:
python_fir = biopac_fir_bandpass(raw, sampling_rate)

In [ ]:
fir_comparison = compare(python_fir, reference["fir"])
fir_comparison

### Read the FIR differences

`max_absolute_error` is the largest sample difference. `rmse` measures the overall error size; `mean_error` shows signed bias. `worst_sample` and `worst_time_seconds` locate the largest difference. Zero errors mean identical values for this recording. Nonzero errors need investigation; neither high correlation nor a visually similar trace establishes equivalence.

Separate the boundaries from the interior next. The Python FIR uses half its coefficient count on each side. That boundary width describes **our filter**, not a verified AcqKnowledge boundary. All samples remain in the overall comparison. Recordings too short to have an interior omit that row.

In [ ]:
edge_samples = min((python_fir_taps - 1) // 2, len(raw) // 2)
regions = {"beginning": (0, edge_samples),
           "interior": (edge_samples, len(raw) - edge_samples),
           "end": (len(raw) - edge_samples, len(raw))}
region_rows = []
for name, (left, right) in regions.items():
    if right > left:
        errors = python_fir[left:right] - reference["fir"][left:right]
        region_rows.append({"region": name, "start_sample": left,
                            "stop_sample_exclusive": right, "samples": right - left,
                            "max_absolute_error": np.max(np.abs(errors)),
                            "rmse": np.sqrt(np.mean(errors ** 2))})
pd.DataFrame(region_rows)

### Inspect one interval

Start with the first second. Change `start_sample` to inspect the interior or the worst mismatch, then rerun the table and plot cells. For example, use `max(0, int(fir_comparison["worst_sample"]) - 20)` around the worst mismatch, or `max(0, len(raw) - 200)` near the end. This only changes the displayed interval; it never shifts or truncates the signals used for the overall comparison.

In [ ]:
start_sample = 0
samples_to_show = int(sampling_rate)
if not 0 <= start_sample < len(raw) or samples_to_show < 1:
    raise ValueError("Choose a valid start sample and a positive interval length.")
stop_sample = min(start_sample + samples_to_show, len(raw))
indices = np.arange(start_sample, stop_sample)
fir_interval = pd.DataFrame({
    "sample": indices,
    "time_seconds": indices / sampling_rate,
    "python": python_fir[start_sample:stop_sample],
    "acqknowledge": reference["fir"][start_sample:stop_sample],
})
fir_interval["difference"] = fir_interval["python"] - fir_interval["acqknowledge"]
fir_interval.head(20).style.format({"time_seconds": "{:.9g}", "python": "{:.17g}",
                                   "acqknowledge": "{:.17g}", "difference": "{:.17g}"})

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(fir_interval["time_seconds"], fir_interval["acqknowledge"],
             label="AcqKnowledge", color="tab:blue")
axes[0].plot(fir_interval["time_seconds"], fir_interval["python"],
             label="Python", color="tab:orange", linestyle="--")
axes[0].set_ylabel(f"Amplitude ({raw_channel.units})")
axes[0].set_title("FIR comparison — selected interval")
axes[0].legend()
axes[1].plot(fir_interval["time_seconds"], fir_interval["difference"], color="tab:purple")
axes[1].axhline(0, color="gray", linewidth=0.7)
axes[1].set_ylabel(f"Python − AcqKnowledge ({raw_channel.units})")
axes[1].set_xlabel("Time from recording start (seconds)")
fig.tight_layout()
plt.show()

## Stop here: review FIR

Do not run comb or RMS yet. Record the measured errors and what they show before changing a function. No tolerance has been justified yet.

| Difference pattern | Next focused test |
| --- | --- |
| Mostly at the boundaries | Compare boundary extension using the same ramp or nonzero-start signal in both programs. |
| A shifted waveform | Filter a centered impulse in both programs and compare its position and shape. |
| Consistent amplitude difference | Filter the same 264 Hz sine and compare amplitude away from the boundaries. |
| Different shape throughout | Check reported tap count, then compare impulse responses to investigate coefficients/window sampling. |

These are diagnostic clues, not proof of a cause. Use the **same samples and sampling rate** for any controlled test. An impulse tests the filter response; a sine tests gain; a ramp helps distinguish boundary treatment. Only create these additional tests if the first real recording shows a mismatch that needs them.

Agreement for this recording does not establish every possible input case. Keep the native reference and note the software version, settings and observed errors. Change one assumption at a time.

**FIR findings:** pending real reference data.

---
The remaining cells are for later, after we review FIR.

## 3. COMB — continue after reviewing FIR

The reference must retain **FIR and comb results for the same channel**. AcqKnowledge settings: 60 Hz, Q=5, all harmonics up to Nyquist, entire waveform.

First test Python comb on **AcqKnowledge FIR**. This isolates the comb function. Then compare the complete Python sequence to see any error inherited from FIR. The sample table, regional errors and plot below use the **isolated** result.

In [ ]:
comb_index = None  # Reference file's comb result; set only when ready for this step.
reference_channels["comb"] = reference_recording.channels[comb_index]
ch = reference_channels["comb"]
if ch.samples_per_second != sampling_rate or ch.units != raw_channel.units:
    raise ValueError("Comb sampling rate or units differ from raw.")
reference["comb"] = np.asarray(ch.data, dtype=float)
if reference["comb"].shape != reference["fir"].shape or reference["comb"].size == 0:
    raise ValueError("COMB: input and output must have equal nonzero lengths.")
if not np.isfinite(reference["comb"]).all():
    raise ValueError("COMB: reference contains missing or infinite samples.")
pd.DataFrame([{"stage": name, "samples": len(reference[name]),
               "rate_hz": reference_channels[name].samples_per_second,
               "units": reference_channels[name].units}
              for name in ("fir", "comb")])

In [ ]:
def biopac_comb_band_stop(data, sampling_rate, base_hz=60.0, q=5.0):
    """
    Approximates the specified AcqKnowledge Comb Band Stop filter settings.
    Equivalence requires sample-by-sample numerical comparison with AcqKnowledge.

    Apply to the band-pass filtered signal. Cascades IIR notch filters at
    base_hz and its integer multiples strictly below Nyquist.

    Each notch uses the same Q, with bandwidth = center_frequency / Q.
    Returns a new array with the same length as the input.

    Reference:
        AcqKnowledge 5 Software Guide, pp. 337–338: Comb Band Stop.
    """
    nyquist = sampling_rate / 2.0

    if base_hz <= 0 or base_hz >= nyquist:
        raise ValueError(
            f"base_hz ({base_hz} Hz) must be positive and strictly less than "
            f"the Nyquist frequency ({nyquist} Hz)."
        )
    if q <= 0:
        raise ValueError(f"q ({q}) must be positive.")

    # Determine the highest harmonic below Nyquist.
    k_max = int(np.floor(nyquist / base_hz))

    # Exclude a harmonic that falls at or above Nyquist.
    if k_max * base_hz >= nyquist:
        k_max -= 1

    filtered = np.array(data, dtype=float, copy=True)

    # Apply notch filters in ascending frequency order.
    # TODO: verify numerical agreement with AcqKnowledge.
    for k in range(1, k_max + 1):
        frequency = k * base_hz

        # BIOPAC documents a cascade of IIR band-stop filters.
        # SciPy uses second-order notches with 3 numerator and 3 denominator coefficients.
        # TODO: verify coefficients against AcqKnowledge using an impulse response.
        b, a = signal.iirnotch(frequency, q, fs=sampling_rate)

        # Apply one forward pass with zero initial state.
        # TODO: verify AcqKnowledge's processing direction and initial conditions.
        # Forward-backward filtering would also change the magnitude response.
        filtered = signal.lfilter(b, a, filtered)

    return filtered

In [ ]:
isolated_comb = biopac_comb_band_stop(reference["fir"], sampling_rate)

In [ ]:
comb_comparison = compare(isolated_comb, reference["comb"])
comb_comparison

Compare the complete sequence separately. If the isolated result agrees but this one differs, an earlier step contributes to the mismatch.

In [ ]:
python_comb = biopac_comb_band_stop(python_fir, sampling_rate)
compare(python_comb, reference["comb"])

### Inspect regions of the isolated result

For comb, the first/last second are diagnostic intervals only: an IIR filter has no fixed finite boundary width, and transients can last longer. For RMS, separate the first 999 startup samples from complete windows, and inspect the end separately. No region is excluded from the overall comparison.

In [ ]:
edge_samples = min(int(sampling_rate), len(raw) // 2)
regions = {"beginning": (0, edge_samples),
           "interior": (edge_samples, len(raw) - edge_samples),
           "end": (len(raw) - edge_samples, len(raw))}
region_rows = []
for name, (left, right) in regions.items():
    if right > left:
        errors = isolated_comb[left:right] - reference["comb"][left:right]
        region_rows.append({"region": name, "start_sample": left,
                            "stop_sample_exclusive": right, "samples": right - left,
                            "max_absolute_error": np.max(np.abs(errors)),
                            "rmse": np.sqrt(np.mean(errors ** 2))})
pd.DataFrame(region_rows)

### Inspect one COMB interval

Change `start_sample`, then rerun the table and plot. To inspect the worst mismatch use `max(0, int(comb_comparison["worst_sample"]) - 20)`; near the end use `max(0, len(raw) - 200)`. All indices are zero-based; neither array is shifted. This selection affects only the display.

In [ ]:
start_sample = 0
samples_to_show = int(sampling_rate)
if not 0 <= start_sample < len(raw) or samples_to_show < 1:
    raise ValueError("Choose a valid start sample and a positive interval length.")
stop_sample = min(start_sample + samples_to_show, len(raw))
indices = np.arange(start_sample, stop_sample)
comb_interval = pd.DataFrame({
    "sample": indices, "time_seconds": indices / sampling_rate,
    "python": isolated_comb[start_sample:stop_sample],
    "acqknowledge": reference["comb"][start_sample:stop_sample],
})
comb_interval["difference"] = comb_interval["python"] - comb_interval["acqknowledge"]
comb_interval.head(20).style.format({"time_seconds": "{:.9g}", "python": "{:.17g}",
                                     "acqknowledge": "{:.17g}", "difference": "{:.17g}"})

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(comb_interval["time_seconds"], comb_interval["acqknowledge"],
             label="AcqKnowledge", color="tab:blue")
axes[0].plot(comb_interval["time_seconds"], comb_interval["python"],
             label="Python on native input", color="tab:orange", linestyle="--")
axes[0].set_ylabel(f"Amplitude ({raw_channel.units})")
axes[0].set_title("COMB comparison — isolated function, selected interval")
axes[0].legend()
axes[1].plot(comb_interval["time_seconds"], comb_interval["difference"], color="tab:purple")
axes[1].axhline(0, color="gray", linewidth=0.7)
axes[1].set_ylabel(f"Python − AcqKnowledge ({raw_channel.units})")
axes[1].set_xlabel("Time from recording start (seconds)")
fig.tight_layout()
plt.show()

## Stop here: review comb

- Mostly startup error: investigate initial conditions using the same nonzero-start input.
- Timing or waveform differences throughout: compare an impulse response to investigate coefficients and processing direction.
- Frequency-dependent differences: compare the same sine signals near 60 Hz and its harmonics to investigate notch width and gain.

These are clues, not proof. Inspect the isolated result before attributing a difference to comb. Record observations; do not change several assumptions at once or widen a tolerance to hide the mismatch.

**Comb findings:** pending real reference data. Continue to RMS only after reviewing this step.

## 4. RMS — continue after reviewing comb

The reference must retain **comb and RMS results for the same channel**. AcqKnowledge settings: Average over samples, 1000 samples, Root mean square, Remove baseline unchecked, entire waveform.

First test Python RMS on **AcqKnowledge comb**. Then compare the complete Python sequence. The table and plot below use the **isolated** result. The fixed denominator of 999 is still provisional.

In [ ]:
rms_index = None  # Reference file's RMS result; set only when ready for this step.
reference_channels["rms"] = reference_recording.channels[rms_index]
ch = reference_channels["rms"]
if ch.samples_per_second != sampling_rate or ch.units != raw_channel.units:
    raise ValueError("RMS sampling rate or units differ from raw.")
reference["rms"] = np.asarray(ch.data, dtype=float)
if reference["rms"].shape != reference["comb"].shape or reference["rms"].size == 0:
    raise ValueError("RMS: input and output must have equal nonzero lengths.")
if not np.isfinite(reference["rms"]).all():
    raise ValueError("RMS: reference contains missing or infinite samples.")
pd.DataFrame([{"stage": name, "samples": len(reference[name]),
               "rate_hz": reference_channels[name].samples_per_second,
               "units": reference_channels[name].units}
              for name in ("comb", "rms")])

In [ ]:
def biopac_moving_rms(data, window_samples=1000):
    """
    Approximates AcqKnowledge Transform > Integrate > Root mean square.

    Uses a trailing window of 1000 samples by default, matching our protocol.
    Each output includes the current sample and preceding samples.
    At the beginning, only available samples contribute to the sum.

    Uses a provisional fixed denominator of window_samples - 1,
    including during startup. AcqKnowledge's manual gives conflicting
    descriptions of the denominator.

    Expects a nonempty 1D signal.
    Returns a new array with the same length, leaving the input unchanged.
    Exact AcqKnowledge equivalence requires numerical comparison.
    """
    if window_samples < 2:
        raise ValueError("window_samples must be at least 2.")

    squared = np.asarray(data, dtype=float) ** 2

    # Trailing sums: current sample plus up to window_samples - 1 previous samples.
    # Taking the first len(data) outputs preserves timing and signal length.
    # No reflected samples are introduced at the beginning.
    window_sums = np.convolve(
        squared, np.ones(window_samples), mode="full"
    )[:len(squared)]

    # Provisional fixed denominator, including for incomplete startup windows.
    # TODO: verify both startup values and s versus s-1 against AcqKnowledge.
    # For a full 1000-sample window of alternating +1/-1:
    # denominator 1000 -> 1.000000; denominator 999 -> 1.000500375...
    mean_squared = window_sums / (window_samples - 1)

    return np.sqrt(mean_squared)

In [ ]:
isolated_rms = biopac_moving_rms(reference["comb"])

In [ ]:
rms_comparison = compare(isolated_rms, reference["rms"])
rms_comparison

Compare the complete sequence separately. If the isolated result agrees but this one differs, an earlier step contributes to the mismatch.

In [ ]:
python_rms = biopac_moving_rms(python_comb)
compare(python_rms, reference["rms"])

### Inspect regions of the isolated result

For comb, the first/last second are diagnostic intervals only: an IIR filter has no fixed finite boundary width, and transients can last longer. For RMS, separate the first 999 startup samples from complete windows, and inspect the end separately. No region is excluded from the overall comparison.

In [ ]:
startup_end = min(999, len(raw))
end_start = max(startup_end, len(raw) - 1000)
regions = {"startup": (0, startup_end),
           "complete_windows_interior": (startup_end, end_start),
           "complete_windows_end": (end_start, len(raw))}
region_rows = []
for name, (left, right) in regions.items():
    if right > left:
        errors = isolated_rms[left:right] - reference["rms"][left:right]
        region_rows.append({"region": name, "start_sample": left,
                            "stop_sample_exclusive": right, "samples": right - left,
                            "max_absolute_error": np.max(np.abs(errors)),
                            "rmse": np.sqrt(np.mean(errors ** 2))})
pd.DataFrame(region_rows)

### Inspect one RMS interval

Change `start_sample`, then rerun the table and plot. To inspect the worst mismatch use `max(0, int(rms_comparison["worst_sample"]) - 20)`; near the end use `max(0, len(raw) - 200)`. All indices are zero-based; neither array is shifted. This selection affects only the display.

In [ ]:
start_sample = 0
samples_to_show = int(sampling_rate)
if not 0 <= start_sample < len(raw) or samples_to_show < 1:
    raise ValueError("Choose a valid start sample and a positive interval length.")
stop_sample = min(start_sample + samples_to_show, len(raw))
indices = np.arange(start_sample, stop_sample)
rms_interval = pd.DataFrame({
    "sample": indices, "time_seconds": indices / sampling_rate,
    "python": isolated_rms[start_sample:stop_sample],
    "acqknowledge": reference["rms"][start_sample:stop_sample],
})
rms_interval["difference"] = rms_interval["python"] - rms_interval["acqknowledge"]
rms_interval.head(20).style.format({"time_seconds": "{:.9g}", "python": "{:.17g}",
                                     "acqknowledge": "{:.17g}", "difference": "{:.17g}"})

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(rms_interval["time_seconds"], rms_interval["acqknowledge"],
             label="AcqKnowledge", color="tab:blue")
axes[0].plot(rms_interval["time_seconds"], rms_interval["python"],
             label="Python on native input", color="tab:orange", linestyle="--")
axes[0].set_ylabel(f"Amplitude ({raw_channel.units})")
axes[0].set_title("RMS comparison — isolated function, selected interval")
axes[0].legend()
axes[1].plot(rms_interval["time_seconds"], rms_interval["difference"], color="tab:purple")
axes[1].axhline(0, color="gray", linewidth=0.7)
axes[1].set_ylabel(f"Python − AcqKnowledge ({raw_channel.units})")
axes[1].set_xlabel("Time from recording start (seconds)")
fig.tight_layout()
plt.show()

### Compare the two fixed RMS denominators

Use the **same native comb input**, trailing window and 1000-sample width for both candidates. Only the denominator changes: 1000 versus 999. Compare complete windows separately from startup. The first complete trailing window ends at index **999**.

A smaller error is evidence, not an automatic choice: alignment and startup rules must also agree. This tests two **fixed** denominators, not an adaptive startup denominator. The cleaning function remains unchanged.

In [ ]:
window_sums = np.convolve(reference["comb"] ** 2, np.ones(1000), mode="full")[:len(raw)]
denominator_rows = []
for denominator in (1000, 999):
    candidate = np.sqrt(window_sums / denominator)
    for region, left, right in [("startup", 0, min(999, len(raw))),
                                ("complete_windows", 999, len(raw))]:
        if right > left:
            errors = candidate[left:right] - reference["rms"][left:right]
            denominator_rows.append({"denominator": denominator, "region": region,
                                     "samples": right - left,
                                     "max_absolute_error": np.max(np.abs(errors)),
                                     "rmse": np.sqrt(np.mean(errors ** 2))})
pd.DataFrame(denominator_rows)

## Stop here: review RMS

- Differences only in the first 999 samples: investigate startup handling.
- A shifted envelope: investigate trailing versus centered alignment with a single impulse at the same position in both programs.
- A small consistent amplitude difference in complete windows: inspect the two denominator candidates above.

If needed, apply **only RMS** to an identical sequence of alternating +1/-1 samples in both programs. A complete 1000-sample window gives 1.0 with denominator 1000, or approximately 1.000500375 with denominator 999. Read saved values at full precision and inspect startup separately. Filtering that test signal first would change the input and invalidate these expected values.

**RMS findings:** pending real reference data. No automatic PASS threshold has been assigned. Agreement on one recording does not prove every input or boundary case.

The isolated comparison feeds the **AcqKnowledge preceding stage** into the Python function. If that agrees but the complete sequence differs, an earlier step contributes to the mismatch. Agreement on one recording supports that recording only; it does not prove every boundary case.

## 5. Inspect individual samples
Choose a stage and sample interval. Inspect the beginning, around a burst, near the reported worst sample, and the end. Do not shift signals to make them agree: timing is part of this test. For RMS, the first 999 samples deserve separate attention because startup and the denominator are unresolved.

In [ ]:
stage = "fir"
start = 0
stop = 20
python_stages = {"fir": python_fir, "comb": python_comb, "rms": python_rms}
actual = python_stages[stage]
expected = reference[stage]
pd.DataFrame({
    "sample": np.arange(len(actual))[start:stop],
    "python": actual[start:stop],
    "acqknowledge": expected[start:stop],
    "difference": (actual - expected)[start:stop],
})

## Record what needs changing

- AcqKnowledge version: …
- FIR reported coefficient count: …
- FIR: rounding/odd count, window sampling, gain and boundaries remain to verify.
- Comb: coefficients, processing direction and initial conditions remain to verify.
- RMS: fixed 999 versus 1000 denominator, startup values and timing remain to verify.

Keep the reference `.acq` and these results. Change only the first disagreeing function, restart the kernel and rerun the comparisons. No files are written by this notebook.